In [5]:
from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import io
from pathlib import Path
from getpass import getpass

import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()  # reads KALSHI_* settings from a local .env

data_path = Path("kalshi_events.parquet")

if data_path.exists():
    df = pd.read_parquet(data_path)
    print("Loaded the local copy.")
else:
    # Credentials come from a local .env file (never commit them): KALSHI_S3_BASE, KALSHI_USER, KALSHI_PASSWORD
    url = os.environ["KALSHI_S3_BASE"].rstrip("/") + "/events-2026-10-09T23.parquet"

    response = requests.get(
        url,
        auth=(os.environ["KALSHI_USER"], os.environ["KALSHI_PASSWORD"]),
        timeout=120,
    )
    response.raise_for_status()

    df = pd.read_parquet(io.BytesIO(response.content))
    df.to_parquet(data_path, index=False)
    print("Downloaded and saved the file.")

Loaded the local copy.


In [3]:
df["timestamp"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)

print(f"Loaded {len(df):,} rows.")
display(df.head(10))

Loaded 628,513 rows.


,ts_ms,asset_id,market,side,price,size,timestamp
0,1791589563481,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,BUY,0.41,2.60,2026-10-09 23:46:03.481+00:00
1,1791589565419,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,SELL,0.89,0.00,2026-10-09 23:46:05.419+00:00
2,1791589565420,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,SELL,0.90,2.01,2026-10-09 23:46:05.420+00:00
3,1791589598438,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,BUY,0.41,67.60,2026-10-09 23:46:38.438+00:00
4,1791589779889,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,SELL,0.80,0.00,2026-10-09 23:49:39.889+00:00
5,1791589779899,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,BUY,0.20,0.00,2026-10-09 23:49:39.899+00:00
6,1791589779914,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,SELL,0.89,48.00,2026-10-09 23:49:39.914+00:00
7,1791589779914,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,SELL,0.90,48.01,2026-10-09 23:49:39.914+00:00
8,1791589779932,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,BUY,0.18,1276.00,2026-10-09 23:49:39.932+00:00
9,1791589779943,EVSHARE-30JAN-30,92fb2641-61bc-4dc0-8f3a-35f440ab149c,SELL,0.80,401.00,2026-10-09 23:49:39.943+00:00


In [8]:
# Inspect the data
df.info()

display(df["side"].value_counts(dropna=False).rename("row_count"))

print("Earliest timestamp:", df["timestamp"].min())
print("Latest timestamp:", df["timestamp"].max())


# %% Plot inside the Interactive Window
counts = df["asset_id"].value_counts().head(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
counts.plot.barh(ax=ax, color="steelblue")

ax.set(
    title="Top 10 assets by number of recorded rows",
    xlabel="Recorded rows",
    ylabel="Asset ID",
)

plt.tight_layout()
plt.show()

<class 'pandas.DataFrame'>
RangeIndex: 628513 entries, 0 to 628512
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   ts_ms     628513 non-null  int64  
 1   asset_id  628513 non-null  str    
 2   market    628513 non-null  str    
 3   side      628513 non-null  str    
 4   price     628513 non-null  float64
 5   size      628513 non-null  float64
dtypes: float64(2), int64(1), str(3)
memory usage: 69.6 MB


side
SELL    318527
BUY     309986
Name: row_count, dtype: int64

KeyError: 'timestamp'

In [5]:
%pip install sqlalchemy psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 29.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine, URL, text

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.environ["KALSHI_USER"],      # from a local .env (never commit it)
    password=os.environ["KALSHI_PASSWORD"],
    host=os.environ["KALSHI_DB_HOST"],
    port=5432,
    database="kalshi",
)

engine = create_engine(
    connection_url,
    connect_args={
        "sslmode": "require",
        "connect_timeout": 15,
    },
)

# Test the connection
with engine.connect() as connection:
    database_name = connection.execute(
        text("SELECT current_database()")
    ).scalar()

print("Connected to:", database_name)

Connected to: kalshi


In [10]:
with engine.connect() as connection:
    markets_df = pd.read_sql_query(
        text("""
            SELECT slug, title, best_bid, best_ask
            FROM public.markets
            LIMIT 10
        """),
        connection,
    )

display(markets_df)

,slug,title,best_bid,best_ask
0,KXMVECROSSCATEGORY-S2026B46516D78F3-F35D21B92EF,"yes Manchester United wins 1st Half,yes Manche...",0.0,0.0
1,KXMVECROSSCATEGORY-S2026B6F05E5740E-2B150EB30EB,"yes Jonathan Taylor: 60+,yes Kenneth Walker: 5...",0.0,0.0
2,KXMVECROSSCATEGORY-S2026B92125AF3A6-53A93ED8354,"yes Over 1.5 goals scored,yes Over 1.5 goals s...",0.0,0.0
3,KXMVECROSSCATEGORY-S2026B98FE4D811A-4E505C437A2,"yes Joe Burrow: 200+,yes Jordan Love: 200+,yes...",0.0,0.0
4,KXMVECROSSCATEGORY-S2026BA88ACCB3A2-C09559B412C,"yes Atlanta,yes Jordan Love: 250+,yes Caleb Wi...",0.0,0.0
5,KXMVECROSSCATEGORY-S2026BB2DBAB8B77-FA870909DBA,"yes Cleveland,yes Tampa Bay,yes Chicago C,yes ...",0.0,1.0
6,KXMVECROSSCATEGORY-S2026BDEC8BE0B61-0774F06A166,"yes Philadelphia,yes Tampa Bay,yes Los Angeles...",0.0,0.0
7,KXMVECROSSCATEGORY-S2026C245BC30B1C-09E932D064F,"yes Philadelphia,yes Kansas City,yes Chicago C...",0.0,1.0
8,KXMVECROSSCATEGORY-S2026C24DD3B5D87-08FC4F6FFD8,"yes Olimpia,yes Over 0.5 goals scored,yes Over...",0.0,1.0
9,KXMVECROSSCATEGORY-S2026E735F5F2D1B-DA63C012016,"yes Harvard,yes Indiana,yes Indiana over 20.5 ...",0.0,1.0


In [3]:
with engine.connect() as connection:
    columns_df = pd.read_sql_query(
        text("""
            SELECT table_name, column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name IN ('markets', 'events', 'hourly_candles')
            ORDER BY table_name, ordinal_position
        """),
        connection,
    )

display(columns_df)

,table_name,column_name,data_type
0,events,slug,text
1,events,title,text
2,events,ticker,text
3,events,start_date,timestamp with time zone
4,events,end_date,timestamp with time zone
...,...,...,...
58,markets,volume_unit,text
59,markets,source_updated_at,timestamp with time zone
60,markets,observed_at,timestamp with time zone
61,markets,raw_payload,jsonb


In [4]:
engine.dispose()